In [11]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import numpy as np
import random
from collections import deque
from snakeGameVAI import SnakeGameAI, WIDTH, HEIGHT, GRID_SIZE
import os
import pygame

In [12]:
class SnakeCNN(nn.Module):
    def __init__(self, w, h):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        # Calculate flattened size: 32 channels * width * height
        # since the screen w,h is 600, 400 the final feature map after this layer will be 32 x 30 x 20
        # stride 1 and padding 1 keeps the dimensions same
        self.fc1 = nn.Linear(32 * w * h, 128)
        self.fc2 = nn.Linear(128, 4) # 4 Actions

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = x.view(x.size(0), -1) 
        x = F.relu(self.fc1(x))
        return self.fc2(x)

In [13]:
class Agent:
    def __init__(self, w, h):
        self.n_games = 0
        self.epsilon = 1.0  # Randomness
        self.gamma = 0.9    # Discount rate
        self.memory = deque(maxlen=100_000)
        self.record = 0
        self.model = SnakeCNN(w, h).to('cuda') # Move to your 4060
        self.optimizer = optim.Adam(self.model.parameters(), lr=0.001)
        self.criterion = nn.MSELoss()
        self.model_path = 'model/snake_cnn.pth'
        self.load_model()
    
    def save_model(self):
        """Save model, optimizer state, and training progress"""
        os.makedirs(os.path.dirname(self.model_path), exist_ok=True)
        checkpoint = {
            'model_state_dict': self.model.state_dict(),
            'optimizer_state_dict': self.optimizer.state_dict(),
            'n_games': self.n_games,
            'epsilon': self.epsilon,
            'record': self.record
        }
        torch.save(checkpoint, self.model_path)
        print(f'Model saved to {self.model_path}')
        
    def load_model(self):
        """Load model if it exists"""
        if os.path.exists(self.model_path):
            print(f'Loading model from {self.model_path}...')
            checkpoint = torch.load(self.model_path)
            self.model.load_state_dict(checkpoint['model_state_dict'])
            self.optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
            self.n_games = checkpoint['n_games']
            self.epsilon = checkpoint['epsilon']
            self.record = checkpoint['record']
            print(f'Resumed: Game {self.n_games}, Record: {self.record}, Epsilon: {self.epsilon:.2f}')
        else:
            print('No saved model found. Starting fresh.')
            
    def get_action(self, state):
        # Exploration vs Exploitation
        self.epsilon = max(0.01, 0.8 - (self.n_games / 200)) # Decay
        if random.random() < self.epsilon:
            return random.randint(0, 3)
        
        # converting a 2d array state grid (30x20) to a 4d input for cnn (batch, channel, height, width) - first unsqueeze adds channel second batch
        state_tensor = torch.tensor(state, dtype=torch.float).unsqueeze(0).unsqueeze(0).to('cuda')
        prediction = self.model(state_tensor)
        return torch.argmax(prediction).item()

    def train_short_memory(self, state, action, reward, next_state, done):
        self.train_step(state, action, reward, next_state, done)

    def train_step(self, state, action, reward, next_state, done):
        state = torch.tensor(np.array(state), dtype=torch.float).to('cuda')
        next_state = torch.tensor(np.array(next_state), dtype=torch.float).to('cuda')
        action = torch.tensor(action, dtype=torch.long).to('cuda')
        reward = torch.tensor(reward, dtype=torch.float).to('cuda')

        if len(state.shape) == 2: # Add batch dimension if single sample
            state = state.unsqueeze(0).unsqueeze(0)
            next_state = next_state.unsqueeze(0).unsqueeze(0)
            action = action.unsqueeze(0)
            reward = reward.unsqueeze(0)
            done = (done, )
        else:
            state = state.unsqueeze(1)      # -> (1000, 1, 20, 30)
            next_state = next_state.unsqueeze(1)

        # Bellman Equation Logic
        pred = self.model(state)
        target = pred.clone()
        for idx in range(len(done)):
            Q_new = reward[idx]
            if not done[idx]:
                # predict the sanke ais next move - apply the equation to calculate q value
                # next_state[idx].unsqueeze(0) - unsqueeze to add batch dimension - notice we are running this through the model
                Q_new = reward[idx] + self.gamma * torch.max(self.model(next_state[idx].unsqueeze(0)))
            target[idx][action[idx]] = Q_new

        self.optimizer.zero_grad()
        # comput how far the ai will be in next state from where it is now
        loss = self.criterion(target, pred)
        loss.backward()
        self.optimizer.step()

In [14]:
def train():
    game = SnakeGameAI(render_speed=10) # Your Pygame class modified to return state/reward
    agent = Agent(WIDTH//GRID_SIZE, HEIGHT//GRID_SIZE)
    try:
        while True:
            # 1. Get current state
            state_old = game.get_state() 
            
            # 2. Get move
            move = agent.get_action(state_old)
            
            # 3. Perform move and get new state
            reward, done, score = game.play_step(move)
            state_new = game.get_state()
            
            # 4. Train short memory (current move)
            agent.train_short_memory(state_old, move, reward, state_new, done)
            
            # 5. Store in memory
            agent.memory.append((state_old, move, reward, state_new, done))
            
            if done:
                game.reset()
                agent.n_games += 1
                # 6. Train long memory (Experience Replay)
                if len(agent.memory) > 1000:
                    mini_batch = random.sample(agent.memory, 1000)
                    states, actions, rewards, next_states, dones = zip(*mini_batch)
                    agent.train_step(states, actions, rewards, next_states, dones)
                if score > agent.record:
                    agent.record = score
                    agent.save_model()  # Save on new record
                    print(f'🎉 New Record: {score}!')
                if agent.n_games % 50 == 0:
                        agent.save_model()
                
                print(f'Game {agent.n_games}, Score: {score}, Record: {agent.record}, Epsilon: {agent.epsilon:.2f}')
    except KeyboardInterrupt:
        # Graceful exit - save before quitting
        print('\n\nTraining interrupted! Saving model...')
        agent.save_model()
        pygame.quit()
        print('Goodbye!')

In [17]:
pygame.init()
train()

No saved model found. Starting fresh.
Updating UI
Drawing grid and elements
Drawing food and snake
Updating UI
Drawing grid and elements
Drawing food and snake
Game 1, Score: 0, Record: 0, Epsilon: 0.80
Updating UI
Drawing grid and elements
Drawing food and snake
Updating UI
Drawing grid and elements
Drawing food and snake
Game 2, Score: 0, Record: 0, Epsilon: 0.80
Updating UI
Drawing grid and elements
Drawing food and snake
Updating UI
Drawing grid and elements
Drawing food and snake
Game 3, Score: 0, Record: 0, Epsilon: 0.79
Updating UI
Drawing grid and elements
Drawing food and snake
Updating UI
Drawing grid and elements
Drawing food and snake
Game 4, Score: 0, Record: 0, Epsilon: 0.79
Updating UI
Drawing grid and elements
Drawing food and snake
Updating UI
Drawing grid and elements
Drawing food and snake
Game 5, Score: 0, Record: 0, Epsilon: 0.78
Updating UI
Drawing grid and elements
Drawing food and snake
Updating UI
Drawing grid and elements
Drawing food and snake
Game 6, Score: 

SystemExit: 

m:\anaconda\envs\alex\lib\site-packages\IPython\core\interactiveshell.py:3587: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
